In [20]:
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
%matplotlib inline
from tensorflow import keras
import seaborn as sns 

In [21]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
train.head()

,id,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,0,D109,HARD,Canadian Grand Prix,2022,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,1,D086,HARD,Dutch Grand Prix,2025,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,2,ZON,HARD,Austrian Grand Prix,2022,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,3,SPE,MEDIUM,Pre-Season Testing,2023,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,4,D019,HARD,Azerbaijan Grand Prix,2022,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [22]:
train.drop(columns=['id', 'Race', 'Year', 'Driver'],inplace=True,  axis = 1)

In [23]:
train.head()

,Compound,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,HARD,0,50,2,39.0,8,78.491,-7.564,21.019,0.714286,5.0,1.0
1,HARD,1,27,2,7.0,4,75.095,-32.617,-223.207,0.346154,-3.0,0.0
2,HARD,0,59,3,22.0,13,70.945,-7.540,-100.529,0.819444,3.0,1.0
3,MEDIUM,0,2,1,2.0,7,94.361,-7.324,-7.324,0.076923,0.0,0.0
4,HARD,1,26,3,6.0,2,107.878,8.965,-14.139,0.361111,3.0,0.0


In [24]:
print(train.isnull().sum())

Compound                  0
PitStop                   0
LapNumber                 0
Stint                     0
TyreLife                  0
Position                  0
LapTime (s)               0
LapTime_Delta             0
Cumulative_Degradation    0
RaceProgress              0
Position_Change           0
PitNextLap                0
dtype: int64


In [25]:
from sklearn.model_selection import train_test_split
X = train.drop('PitNextLap', axis = 1)
y =train['PitNextLap']

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size = 0.2, random_state = 42)
X_train.shape, X_test.shape

((351312, 11), (87828, 11))

In [26]:
X_train.columns

Index(['Compound', 'PitStop', 'LapNumber', 'Stint', 'TyreLife', 'Position',
       'LapTime (s)', 'LapTime_Delta', 'Cumulative_Degradation',
       'RaceProgress', 'Position_Change'],
      dtype='object')

In [27]:
# Create Column Transformer with 3 types of transformers
cat_features = X.select_dtypes(include="object").columns
num_features = X.select_dtypes(exclude="object").columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder(drop='first')

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features)
    ]
)

X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout

model = Sequential()

model.add(Dense(64, activation= 'relu', input_shape = (X_train.shape[1],)))
model.add(Dropout(.2))


model.add(Dense(32, activation= 'relu'))
model.add(Dropout(.2))

model.add(Dense(1, activation ='sigmoid'))

C:\Users\User\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [29]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',  # binary target
    metrics=['accuracy']
)

In [30]:
history = model.fit(X_train, y_train,
                    validation_data = (X_test, y_test),
                    epochs = 30,
                    batch_size = 128)

Epoch 1/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.8586 - loss: 0.3263 - val_accuracy: 0.8741 - val_loss: 0.2903
Epoch 2/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 11s 4ms/step - accuracy: 0.8727 - loss: 0.2957 - val_accuracy: 0.8770 - val_loss: 0.2816
Epoch 3/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 10s 4ms/step - accuracy: 0.8758 - loss: 0.2890 - val_accuracy: 0.8791 - val_loss: 0.2756
Epoch 4/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.8772 - loss: 0.2842 - val_accuracy: 0.8807 - val_loss: 0.2715
Epoch 5/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.8779 - loss: 0.2805 - val_accuracy: 0.8818 - val_loss: 0.2672
Epoch 6/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - accuracy: 0.8788 - loss: 0.2773 - val_accuracy: 0.8819 - val_loss: 0.2635
Epoch 7/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.8790 - loss: 0.2758 - val_accuracy: 0.8823 - val_loss: 0.2627
Epoch 8/30
2745/2745 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - accuracy: 0.8795 - loss: 0.273

In [31]:
loss, acc = model.evaluate(X_test, y_test)
print("Validation Accuracy:", acc)

2745/2745 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.8842 - loss: 0.2551
Validation Accuracy: 0.8842169046401978


In [32]:
X_test_scaled = numeric_transformer.transform(X_test)
y_pred_prob = model.predict(X_test_scaled)

# Convert probabilities to binary
y_pred = (y_pred_prob > 0.5).astype(int)

# Prepare submission
if len(y_pred) != len(test):
    test_features = test.drop(columns=["id", "Race", "Year", "Driver"], errors="ignore")
    test_processed = preprocessor.transform(test_features)
    y_pred_prob = model.predict(test_processed)
    y_pred = (y_pred_prob > 0.5).astype(int)

submission = pd.DataFrame({
    "id": test["id"].values,
    "PitNextLap": y_pred.flatten()
})
submission.to_csv("submission.csv", index=False)

NotFittedError: This StandardScaler instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.